In [1]:
import tensorflow as tf
import tqdm

In [2]:
## Downloading GPT-2 Weights ##
import os
import requests
import json
import numpy as np

def download_and_load_gpt2(model_size, model_dir):
  allowed_sizes = ("124M", "355M", "774M", "1558M")
  if model_size not in allowed_sizes:
    raise ValueError(f"model size must be one of: {allowed_sizes}")
  model_dir = os.path.join(model_dir, model_size)
  if not os.path.exists(model_dir):
    os.makedirs(model_dir)
  base_url = "https://openaipublic.blob.core.windows.net/gpt-2/models"
  filenames = [
      "checkpoint", "encoder.json", "hparams.json", "model.ckpt.data-00000-of-00001",
      "model.ckpt.index", "model.ckpt.meta", "vocab.bpe"
  ]
  for filename in filenames:
    file_url = os.path.join(base_url, model_size, filename)
    file_path = os.path.join(model_dir, filename)
    download_file(file_url, file_path)

  hparams_file_path = os.path.join(model_dir, "hparams.json")
  if not os.path.exists(hparams_file_path):
    raise FileNotFoundError(f"Failed to download hparams.json to {hparams_file_path}. Please check network connection and permissions.")

  tf_ckpt_path = tf.train.latest_checkpoint(model_dir)
  if tf_ckpt_path is None:
    raise ValueError(f"No TensorFlow checkpoint found in {model_dir}. Please ensure checkpoint file is downloaded correctly.")

  settings = json.load(open(hparams_file_path))
  params = load_gpt2_params_from_tf_ckpts(tf_ckpt_path, settings)

  return settings, params

def download_file(url, destination):
  try:
    # Removed verify=False to enable SSL certificate verification
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

    file_size = int(response.headers.get("content-length", 0))

    # Ensure parent directory exists
    os.makedirs(os.path.dirname(destination), exist_ok=True)

    block_size = 1024
    progress_bar_description = url.split("/")[-1]

    if file_size == 0 and 'content-length' not in response.headers:
        print(f"Warning: No 'content-length' header for {url}. Assuming content might be empty or unknown size.")

    with tqdm.tqdm(total=file_size, unit="iB", unit_scale=True, desc=progress_bar_description) as progress_bar:
      with open(destination, "wb") as file:
        for chunk in response.iter_content(block_size):
          if chunk: # Filter out keep-alive chunks
            progress_bar.update(len(chunk))
            file.write(chunk)

    # Explicit checks after writing to confirm file creation and content
    if not os.path.exists(destination):
        raise IOError(f"File {destination} was not created after download.")
    if os.path.getsize(destination) == 0 and file_size > 0:
        raise IOError(f"File {destination} was created empty, but expected {file_size} bytes.")

    print(f"Successfully downloaded {url} to {destination}. File size: {os.path.getsize(destination)} bytes.")

  except requests.exceptions.HTTPError as e:
    print(f"HTTP Error downloading file from {url}: {e}")
    raise # Re-raise the exception for upstream handling
  except requests.exceptions.RequestException as e:
    print(f"Network Error downloading file from {url}: {e}")
    raise # Re-raise the exception
  except Exception as e:
    print(f"An unexpected error occurred during file download to {destination}: {e}")
    raise # Re-raise any other unexpected exception

def load_gpt2_params_from_tf_ckpts(ckpt_path, settings):
  params = {"blocks": [{} for _ in range(settings["n_layer"]) ]}

  for name, _ in tf.train.list_variables(ckpt_path):
    variable_array = np.squeeze(tf.train.load_variable(ckpt_path, name))
    variable_name_parts = name.split("/")[1:]
    target_dict = params
    if variable_name_parts[0].startswith("h"):
      layer_number = int(variable_name_parts[0][1:])
      target_dict = params["blocks"][layer_number]
    for key in variable_name_parts[1:-1]:
      target_dict = target_dict.setdefault(key, {})
    last_day = variable_name_parts[-1]
    target_dict[last_day] = variable_array
  return params


In [3]:
settings, parmas = download_and_load_gpt2(model_size="124M", model_dir="gpt2")

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 85.1kiB/s]


Successfully downloaded https://openaipublic.blob.core.windows.net/gpt-2/models/124M/checkpoint to gpt2/124M/checkpoint. File size: 77 bytes.


encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 4.36MiB/s]


Successfully downloaded https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json to gpt2/124M/encoder.json. File size: 1042301 bytes.


hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 159kiB/s]


...

Successfully downloaded https://openaipublic.blob.core.windows.net/gpt-2/models/124M/model.ckpt.index to gpt2/124M/model.ckpt.index. File size: 5215 bytes.


model.ckpt.meta: 100%|██████████| 471k/471k [00:00<00:00, 2.92MiB/s]


Successfully downloaded https://openaipublic.blob.core.windows.net/gpt-2/models/124M/model.ckpt.meta to gpt2/124M/model.ckpt.meta. File size: 471155 bytes.


vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 2.69MiB/s]


Successfully downloaded https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe to gpt2/124M/vocab.bpe. File size: 456318 bytes.


In [4]:
print("Settings : ", settings)
print("Params : ",parmas)

Streaming output truncated to the last 5000 lines.
       -1.67155147e-01,  6.68368042e-02,  3.94544527e-02, -4.89491187e-02,
        1.24928258e-01, -5.83402207e-03, -1.24973796e-01, -5.93515038e-02,
       -5.28325886e-02, -1.50952443e-01,  4.72278185e-02,  1.00429237e+00,
        1.28995836e-01,  1.44720292e-02,  6.73662871e-02, -3.68661550e-03,
       -2.92272065e-02, -1.82699159e-01,  9.93781015e-02,  2.29054242e-01,
       -7.94954449e-02, -1.74057800e-02, -7.39754736e-02,  5.93156479e-02,
        6.92667365e-02, -1.68846622e-01,  7.79314488e-02,  1.37052625e-01,
        6.28051609e-02, -1.14866264e-01,  5.66228814e-02,  1.18685864e-01,
        6.43390715e-02, -2.98659354e-02, -1.01568423e-01,  9.21916068e-02,
       -2.30538025e-02, -1.97021320e-01, -1.96507454e-01,  1.11167863e-01,
        4.43039555e-03,  1.09657414e-01, -6.70037940e-02,  5.47943264e-02,
       -1.07018817e-02,  1.17700708e+00,  8.19421411e-02, -8.31026435e-02,
       -5.76983131e-02,  9.84845012e-02,  3.63269

In [5]:
## Visualizing Token Embeddings ##
print(parmas["wte"])
print("Shape of Toeken Embeddings : ", parmas["wte"].shape)

[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]
Shape of Toeken Embeddings :  (50257, 768)


In [6]:
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
print(NEW_CONFIG)

{'vocab_size': 50257, 'context_length': 256, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}


In [7]:
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

In [10]:
###  Model Architecture ###
import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )

    def forward(self, x):
        return self.layers(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        # 2*4*768
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x
        # 2*4*768

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [11]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [12]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [13]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [16]:
# device = "cuda" if torch.cuda.is_available else "cpu"
device = "cpu"
load_weights_into_gpt(gpt, parmas)
gpt.to(device);

In [19]:
## Few Helper Functions for Training Loop ##
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    token_ids_list = token_ids.squeeze(0).tolist()
    text = tokenizer.decode(token_ids_list)
    return text

def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probs = torch.nn.functional.softmax(logits, dim=-1)
        idx_next = torch.argmax(probs, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def generate(model, idx, max_new_tokens, context_size, temprature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        if top_k:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                condition=logits < min_val,
                input=torch.tensor(float("-inf")).to(logits.device),
                other=logits
            )

        if temprature > 0.0:
            logits = logits / temprature
            probas = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probas, num_samples=1)

        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [22]:
torch.manual_seed(123)
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temprature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you toward finding an ideal new way to practice something!

What makes us want to be on top of that?


